# Lab 3


In [ ]:

from __future__ import annotations

import json
import os
import subprocess
import time
from pathlib import Path
from typing import Any

import modal

def _resolve_lab3_project_root() -> Path:
    start = Path.cwd().resolve()
    for path in (start, *start.parents):
        data = path / "Data"
        if data.is_dir() and (data / "L3_LR").is_dir():
            return path
    return start


PROJECT_ROOT = _resolve_lab3_project_root()
LOCAL_DATA_ROOT = PROJECT_ROOT / "Data"
LOCAL_RUNS_ROOT = PROJECT_ROOT / "runs"

APP_NAME = "lab3-fsrcnn-residual-modal"
FUNCTION_NAME = "run_fsrcnn_residual_resume_8080_to9100"
MODAL_DATA_VOLUME = os.environ.get("LAB3_FSRCNN_MODAL_DATA_VOLUME", "lab3-data")
MODAL_RUNS_VOLUME = os.environ.get("LAB3_FSRCNN_MODAL_RUNS_VOLUME", "lab3-runs")
MODAL_GPU = os.environ.get("LAB3_FSRCNN_MODAL_GPU", "L40S")
MODAL_TIMEOUT_MINUTES = int(os.environ.get("LAB3_FSRCNN_MODAL_TIMEOUT_MINUTES", "900"))
POLL_INTERVAL_SECONDS = int(os.environ.get("LAB3_FSRCNN_POLL_SECONDS", "120"))
MODAL_DETACH = os.environ.get("LAB3_FSRCNN_MODAL_DETACH", "true").strip().lower() not in {"0", "false", "no"}
SYNC_DATA_TO_VOLUME = os.environ.get("LAB3_FSRCNN_SYNC_DATA", "true").lower() in {"1", "true", "yes"}
FORCE_DATA_SYNC = os.environ.get("LAB3_FSRCNN_FORCE_DATA_SYNC", "false").lower() in {"1", "true", "yes"}
RUN_MODAL_SUBMISSION = os.environ.get("LAB3_FSRCNN_RUN_MODAL", "true").lower() in {"1", "true", "yes"}
LOCAL_SUBMISSION_LAST_PTH = PROJECT_ROOT / "experiments" / "FSRCNNResidual" / "submission" / "last.pth"
FORCE_RESUME_UPLOAD = os.environ.get("LAB3_FSRCNN_FORCE_RESUME_UPLOAD", "false").lower() in {"1", "true", "yes"}

DEFAULT_RESUME_CHECKPOINT = (
    "/mnt/lab3-runs/runs/fsrcnn_residual_submission_epoch8080/checkpoints/last.pth"
)
RUN_CONFIG: dict[str, Any] = {
    "model_id": "fsrcnn_residual_96_40_m8",
    "run_slug": "fsrcnn_residual_96_40_m8_modal_resume8080_to9100_lr300",
    "resume_checkpoint_path": os.environ.get("LAB3_FSRCNN_RESUME_CHECKPOINT", DEFAULT_RESUME_CHECKPOINT),
    "seed": 1337,
    "train_patch_size": 256,
    "eval_size": 256,
    "batch_size": 16,
    "epochs": 9100,
    "lr": 3e-4,
    "min_lr": 1e-5,
    "warmup_epochs": 5,
    "weight_decay": 1e-4,
    "load_optimizer_state": False,
    "num_workers": int(os.environ.get("LAB3_FSRCNN_NUM_WORKERS", "0")),
    "prefetch_factor": 2,
    "use_amp": True,
    "use_ema": True,
    "ema_decay": 0.999,
    "calibration_count": 128,
    "remote_data_root": "/mnt/lab3-data/Data",
    "remote_runs_root": "/mnt/lab3-runs/runs",
}

print(json.dumps({
    "app_name": APP_NAME,
    "function_name": FUNCTION_NAME,
    "modal_gpu": MODAL_GPU,
    "modal_timeout_minutes": MODAL_TIMEOUT_MINUTES,
    "modal_data_volume": MODAL_DATA_VOLUME,
    "modal_runs_volume": MODAL_RUNS_VOLUME,
    "sync_data_to_volume": SYNC_DATA_TO_VOLUME,
    "force_data_sync": FORCE_DATA_SYNC,
    "run_modal_submission": RUN_MODAL_SUBMISSION,
    "config": RUN_CONFIG,
}, indent=2))


In [ ]:

REMOTE_DATA_ROOT = RUN_CONFIG["remote_data_root"]
REMOTE_RUNS_ROOT = RUN_CONFIG["remote_runs_root"]
print({
    "remote_data_root": REMOTE_DATA_ROOT,
    "remote_runs_root": REMOTE_RUNS_ROOT,
    "expected_train_patch_size": RUN_CONFIG["train_patch_size"],
    "expected_eval_size": RUN_CONFIG["eval_size"],
    "modal_data_volume": MODAL_DATA_VOLUME,
    "modal_runs_volume": MODAL_RUNS_VOLUME,
})


## Model and Training


In [ ]:

def run_fsrcnn_pipeline(config: dict[str, Any]) -> dict[str, Any]:
    import csv
    import json
    import math
    import os
    import random
    import shlex
    import subprocess
    import sys
    import time
    from contextlib import nullcontext
    from dataclasses import asdict, dataclass
    from datetime import datetime
    from pathlib import Path
    from typing import Any

    import numpy as np
    import onnx
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset

    try:
        import onnxruntime as ort
    except Exception:
        ort = None

    IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
    BICUBIC_RESAMPLE = getattr(Image, "Resampling", Image).BICUBIC
    FLIP_LEFT_RIGHT = getattr(Image, "Transpose", Image).FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = getattr(Image, "Transpose", Image).FLIP_TOP_BOTTOM
    ROTATE_90 = getattr(Image, "Transpose", Image).ROTATE_90
    ROTATE_180 = getattr(Image, "Transpose", Image).ROTATE_180
    ROTATE_270 = getattr(Image, "Transpose", Image).ROTATE_270

    @dataclass
    class RunConfig:
        model_id: str
        run_slug: str
        resume_checkpoint_path: str
        seed: int = 1337
        train_patch_size: int = 256
        eval_size: int = 256
        batch_size: int = 16
        epochs: int = 9100
        lr: float = 3e-4
        min_lr: float = 1e-5
        warmup_epochs: int = 5
        weight_decay: float = 1e-4
        load_optimizer_state: bool = False
        num_workers: int = 0
        prefetch_factor: int = 2
        use_amp: bool = True
        use_ema: bool = True
        ema_decay: float = 0.999
        calibration_count: int = 128
        remote_data_root: str = "/mnt/lab3-data/Data"
        remote_runs_root: str = "/mnt/lab3-runs/runs"

    def set_global_seed(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def now_stamp() -> str:
        return datetime.now().strftime("%H%M%S_%d%m")

    def make_unique_run_root(runs_root: Path, run_slug: str) -> Path:
        safe_slug = "".join(ch if ch.isalnum() or ch in {"-", "_"} else "_" for ch in run_slug)
        candidate = runs_root / f"{now_stamp()}_{safe_slug}"
        while candidate.exists():
            time.sleep(1.1)
            candidate = runs_root / f"{now_stamp()}_{safe_slug}"
        return candidate

    def write_json(path: Path, payload: dict[str, Any]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    cfg = RunConfig(**config)
    data_root = Path(cfg.remote_data_root)
    runs_root = Path(cfg.remote_runs_root)
    resume_checkpoint_path = Path(cfg.resume_checkpoint_path)

    if not data_root.exists():
        raise FileNotFoundError(f"Modal data root not found: {data_root}. Sync local Data/ to volume {data_root.parent} first.")
    print({"resume_checkpoint_path": str(resume_checkpoint_path), "exists": resume_checkpoint_path.exists()})
    if not resume_checkpoint_path.exists():
        raise FileNotFoundError(
            f"Resume checkpoint not found: {resume_checkpoint_path}. "
            "Set LAB3_FSRCNN_RESUME_CHECKPOINT to the checkpoint path inside the lab3-runs Modal volume."
        )

    runs_root.mkdir(parents=True, exist_ok=True)
    run_root = make_unique_run_root(runs_root, cfg.run_slug)
    checkpoints_dir = run_root / "checkpoints"
    exports_dir = run_root / "exports"
    preview_dir = exports_dir / "val_preview"
    calibration_dir = exports_dir / "calibration"
    summary_path = run_root / "summary.json"
    metrics_csv_path = run_root / "metrics.csv"
    history_json_path = run_root / "history.json"
    run_manifest_path = run_root / "run_manifest.json"
    latest_status_path = run_root / "latest_status.json"
    events_jsonl_path = run_root / "events.jsonl"
    for directory in [run_root, checkpoints_dir, exports_dir, preview_dir, calibration_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    def append_event(event: dict[str, Any]) -> None:
        payload = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S %Z"), **event}
        with events_jsonl_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload, sort_keys=True) + "\n")

    def commit_run_volume(stage: str) -> None:
        volume = globals().get("runs_volume")
        if volume is None:
            return
        try:
            volume.commit()
        except Exception as exc:
            print({"volume_commit_warning": stage, "error": repr(exc)})

    append_event({"status": "initialized", "run_root": str(run_root), "resume_checkpoint_path": str(resume_checkpoint_path)})
    write_json(latest_status_path, {"status": "initialized", "run_root": str(run_root), "resume_checkpoint_path": str(resume_checkpoint_path)})
    commit_run_volume("initialized")

    set_global_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda_enabled = device.type == "cuda"
    gpu_name = torch.cuda.get_device_name(0) if cuda_enabled else None
    if cuda_enabled:
        torch.backends.cudnn.benchmark = True
        if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
            torch.backends.cuda.matmul.allow_tf32 = True

    print({
        "device": str(device),
        "data_root": str(data_root),
        "run_root": str(run_root),
        "gpu_name": gpu_name,
        "config": asdict(cfg),
    })

    SOURCE_TRAIN_SPLITS = (
        ("LR_HR/L3_HR_train1", "L3_LR/L3_L3_train1", "L3_HR_train1"),
        ("LR_HR/L3_HR_train2", "L3_LR/L3_L3_train2", "L3_HR_train2"),
        ("LR_HR/L3_HR_train3", "L3_LR/L3_LR_train3", "L3_HR_train3"),
        ("LR_HR/L3_HR_train4", "L3_LR/L3_LR_train4", "L3_HR_train4"),
        ("LR_HR/L3_HR_train", "L3_LR/L3_LR_train", "L3_HR_train"),
    )
    STANDARD_TRAIN_SPLITS = (
        ("HR_train/HR_train1", "LR_train/LR_train1", "HR_train1"),
        ("HR_train/HR_train2", "LR_train/LR_train2", "HR_train2"),
        ("HR_train/HR_train3", "LR_train/LR_train3", "HR_train3"),
        ("HR_train/HR_train4", "LR_train/LR_train4", "HR_train4"),
    )
    VAL_SPLIT_CANDIDATES = (("L3_HR_valid", "L3_LR_valid"), ("HR_val", "LR_val"))

    def scan_image_directory(directory: Path) -> dict[str, Path]:
        if not directory.exists():
            raise FileNotFoundError(f"Missing directory: {directory}")
        return {
            path.stem: path
            for path in directory.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        }

    def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
        hr_map = scan_image_directory(hr_dir)
        lr_map = scan_image_directory(lr_dir)
        shared = sorted(set(hr_map) & set(lr_map))
        if not shared:
            raise FileNotFoundError(f"No paired files found for {split_name}: HR={hr_dir}, LR={lr_dir}")
        hr_only = sorted(set(hr_map) - set(lr_map))
        lr_only = sorted(set(lr_map) - set(hr_map))
        if hr_only or lr_only:
            print({"split": split_name, "hr_only_count": len(hr_only), "lr_only_count": len(lr_only)})
        pairs: list[tuple[Path, Path, str]] = []
        for stem in shared:
            name = stem if split_name == "val" else f"{split_name}/{stem}"
            pairs.append((lr_map[stem], hr_map[stem], name))
        return pairs

    def collect_train_pairs_from(splits: tuple[tuple[str, str, str], ...]) -> list[tuple[Path, Path, str]]:
        pairs: list[tuple[Path, Path, str]] = []
        for hr_rel, lr_rel, split_name in splits:
            hr_dir = data_root / hr_rel
            lr_dir = data_root / lr_rel
            if hr_dir.exists() and lr_dir.exists():
                pairs.extend(collect_split_pairs(hr_dir, lr_dir, split_name))
            else:
                print({"skipped_missing_train_split": split_name, "hr_dir": str(hr_dir), "lr_dir": str(lr_dir)})
        return pairs

    def collect_all_pairs() -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
        train_pairs = collect_train_pairs_from(SOURCE_TRAIN_SPLITS)
        train_layout = "source_lr_hr_l3_lr"
        if not train_pairs:
            train_pairs = collect_train_pairs_from(STANDARD_TRAIN_SPLITS)
            train_layout = "standard_hr_train_lr_train"
        if not train_pairs:
            raise FileNotFoundError(f"No training pairs found under {data_root}")

        val_pairs: list[tuple[Path, Path, str]] = []
        val_layout = ""
        for hr_rel, lr_rel in VAL_SPLIT_CANDIDATES:
            hr_dir = data_root / hr_rel
            lr_dir = data_root / lr_rel
            if hr_dir.exists() and lr_dir.exists():
                val_pairs = collect_split_pairs(hr_dir, lr_dir, "val")
                val_layout = f"{hr_rel}/{lr_rel}"
                break
        if not val_pairs:
            raise FileNotFoundError(f"No validation pairs found under {data_root}")
        print({"train_layout": train_layout, "val_layout": val_layout})
        return train_pairs, val_pairs

    class PairedSRDataset(Dataset):
        def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, patch_size: int, eval_size: int, seed: int):
            self.pairs = pairs
            self.train = train
            self.patch_size = patch_size
            self.eval_size = eval_size
            self.seed = seed
            self.epoch = 0

        def set_epoch(self, epoch: int) -> None:
            self.epoch = epoch

        @staticmethod
        def to_tensor(img: Image.Image) -> torch.Tensor:
            arr = np.asarray(img, dtype=np.float32) / 255.0
            return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

        def augment_pair(self, lr: Image.Image, hr: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
            rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
            width, height = lr.size
            patch = self.patch_size
            if width < patch or height < patch:
                lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
                hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
                width, height = lr.size
            left = rng.randint(0, width - patch)
            top = rng.randint(0, height - patch)
            box = (left, top, left + patch, top + patch)
            lr = lr.crop(box)
            hr = hr.crop(box)
            if rng.random() < 0.5:
                lr = lr.transpose(FLIP_LEFT_RIGHT)
                hr = hr.transpose(FLIP_LEFT_RIGHT)
            if rng.random() < 0.5:
                lr = lr.transpose(FLIP_TOP_BOTTOM)
                hr = hr.transpose(FLIP_TOP_BOTTOM)
            rot = rng.choice([0, 1, 2, 3])
            if rot:
                rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
                lr = lr.transpose(rotate_ops[rot - 1])
                hr = hr.transpose(rotate_ops[rot - 1])
            return lr, hr

        def __len__(self) -> int:
            return len(self.pairs)

        def __getitem__(self, index: int) -> dict[str, Any]:
            lr_path, hr_path, name = self.pairs[index]
            lr = Image.open(lr_path).convert("RGB")
            hr = Image.open(hr_path).convert("RGB")
            if self.train:
                lr, hr = self.augment_pair(lr, hr, index)
            elif lr.size != (self.eval_size, self.eval_size):
                lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
                hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            return {"lr": self.to_tensor(lr), "hr": self.to_tensor(hr), "name": name}

    train_pairs, val_pairs = collect_all_pairs()
    if not train_pairs or not val_pairs:
        raise ValueError({"train_pairs": len(train_pairs), "val_pairs": len(val_pairs)})

    train_ds = PairedSRDataset(train_pairs, train=True, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
    val_ds = PairedSRDataset(val_pairs, train=False, patch_size=cfg.eval_size, eval_size=cfg.eval_size, seed=cfg.seed)
    loader_kwargs: dict[str, Any] = {"num_workers": cfg.num_workers, "pin_memory": cuda_enabled}
    if cfg.num_workers > 0:
        loader_kwargs["prefetch_factor"] = cfg.prefetch_factor
        loader_kwargs["persistent_workers"] = True
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False, **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False, **loader_kwargs)
    sample = train_ds[0]
    data_summary = {
        "data_root": str(data_root),
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "train_preview": [name for _, _, name in train_pairs[:5]],
        "val_preview": [name for _, _, name in val_pairs[:5]],
        "sample_lr_shape": tuple(sample["lr"].shape),
        "sample_hr_shape": tuple(sample["hr"].shape),
    }
    print(data_summary)
    append_event({"status": "data_ready", "train_pairs": len(train_pairs), "val_pairs": len(val_pairs)})
    write_json(latest_status_path, {"status": "data_ready", "run_root": str(run_root), "data": data_summary})
    assert tuple(sample["lr"].shape) == (3, cfg.eval_size, cfg.eval_size)
    assert tuple(sample["hr"].shape) == (3, cfg.eval_size, cfg.eval_size)

    class FSRCNNResidualNoUpscale(nn.Module):
        """FSRCNN-style residual refiner for 256x256 RGB input/output."""

        def __init__(self, channels: int = 96, shrink_channels: int = 40, mapping_layers: int = 8, slope: float = 0.10):
            super().__init__()
            self.channels = channels
            self.shrink_channels = shrink_channels
            self.mapping_layers = mapping_layers
            self.slope = slope
            self.features = nn.Sequential(*self._feature_layers())
            self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
            self.init_tail_small()

        def _activation(self) -> nn.Module:
            return nn.LeakyReLU(self.slope, inplace=False)

        def _feature_layers(self) -> list[nn.Module]:
            stem = [nn.Conv2d(3, self.channels, 5, padding=2, bias=True), self._activation()]
            shrink = [nn.Conv2d(self.channels, self.shrink_channels, 1, padding=0, bias=True), self._activation()]
            mapping: list[nn.Module] = []
            for _ in range(self.mapping_layers):
                mapping.extend([nn.Conv2d(self.shrink_channels, self.shrink_channels, 3, padding=1, bias=True), self._activation()])
            expand = [nn.Conv2d(self.shrink_channels, self.channels, 1, padding=0, bias=True), self._activation()]
            return [*stem, *shrink, *mapping, *expand]

        def architecture_summary(self) -> dict[str, str | int]:
            return {
                "stem": f"Conv5x5 3->{self.channels} + LeakyReLU",
                "shrink": f"Conv1x1 {self.channels}->{self.shrink_channels} + LeakyReLU",
                "mapping": f"{self.mapping_layers}x Conv3x3 {self.shrink_channels}->{self.shrink_channels} + LeakyReLU",
                "expand": f"Conv1x1 {self.shrink_channels}->{self.channels} + LeakyReLU",
                "tail": f"Conv3x3 {self.channels}->3 residual delta",
                "forward": "output = input + residual_delta",
            }

        def init_tail_small(self) -> None:
            nn.init.kaiming_normal_(self.tail.weight, nonlinearity="linear")
            self.tail.weight.data.mul_(1e-3)
            if self.tail.bias is not None:
                nn.init.zeros_(self.tail.bias)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x + self.tail(self.features(x))

    def count_parameters(module: nn.Module) -> int:
        return int(sum(param.numel() for param in module.parameters()))

    def operator_audit(module: nn.Module) -> dict[str, int]:
        counts: dict[str, int] = {}
        for child in module.modules():
            if len(list(child.children())) != 0:
                continue
            name = child.__class__.__name__
            counts[name] = counts.get(name, 0) + 1
        return dict(sorted(counts.items()))

    class EMA:
        def __init__(self, model: nn.Module, decay: float):
            self.decay = decay
            self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

        @torch.no_grad()
        def update(self, model: nn.Module) -> None:
            for key, value in model.state_dict().items():
                self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

        @torch.no_grad()
        def copy_to(self, model: nn.Module) -> None:
            model.load_state_dict(self.shadow, strict=True)

    def make_grad_scaler(enabled: bool):
        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
            return torch.amp.GradScaler("cuda", enabled=enabled)
        return torch.cuda.amp.GradScaler(enabled=enabled)

    def autocast_context(device_type: str, enabled: bool):
        if not enabled:
            return nullcontext()
        if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
            return torch.amp.autocast(device_type=device_type, enabled=enabled)
        return torch.cuda.amp.autocast(enabled=enabled)

    @torch.no_grad()
    def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
        mse = F.mse_loss(pred, target, reduction="none").mean(dim=(1, 2, 3))
        psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-12))
        return float(psnr.mean().item())

    def compute_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        mse = F.mse_loss(pred, hr)
        residual_l1 = F.l1_loss(pred - lr, hr - lr)
        return 0.8 * mse + 0.2 * residual_l1

    def lr_at_epoch(epoch_idx: int, *, base_lr: float, min_lr: float, warmup_epochs: int, total_epochs: int, schedule_start_epoch: int = 0) -> float:
        step = epoch_idx - schedule_start_epoch + 1
        span = max(1, total_epochs - schedule_start_epoch)
        if warmup_epochs > 0 and step <= warmup_epochs:
            return base_lr * step / warmup_epochs
        progress = (step - warmup_epochs) / max(1, span - warmup_epochs)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr + cosine * (base_lr - min_lr)

    def evaluate_metrics(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
        model.eval()
        pred_meter = 0.0
        input_meter = 0.0
        count = 0
        for batch in loader:
            lr = batch["lr"].to(device, non_blocking=cuda_enabled)
            hr = batch["hr"].to(device, non_blocking=cuda_enabled)
            pred = model(lr).clamp(0.0, 1.0)
            pred_meter += batch_psnr(pred, hr)
            input_meter += batch_psnr(lr, hr)
            count += 1
        val_psnr = pred_meter / max(count, 1)
        input_psnr = input_meter / max(count, 1)
        return {"val_psnr": float(val_psnr), "input_psnr": float(input_psnr), "delta_psnr": float(val_psnr - input_psnr)}

    def tensor_to_uint8_image(x: torch.Tensor) -> Image.Image:
        x = x.detach().cpu().clamp(0.0, 1.0)
        arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
        return Image.fromarray(arr)

    @torch.no_grad()
    def write_val_preview(model: nn.Module, loader: DataLoader, out_dir: Path, device: torch.device) -> dict[str, str]:
        out_dir.mkdir(parents=True, exist_ok=True)
        batch = next(iter(loader))
        lr = batch["lr"].to(device, non_blocking=cuda_enabled)
        hr = batch["hr"].to(device, non_blocking=cuda_enabled)
        pred = model(lr).clamp(0.0, 1.0)
        lr_path = out_dir / "val_preview_input.png"
        pred_path = out_dir / "val_preview_pred.png"
        hr_path = out_dir / "val_preview_target.png"
        tensor_to_uint8_image(lr[0]).save(lr_path)
        tensor_to_uint8_image(pred[0]).save(pred_path)
        tensor_to_uint8_image(hr[0]).save(hr_path)
        name = batch["name"][0] if "name" in batch else "unknown"
        return {"sample_name": str(name), "input": str(lr_path), "prediction": str(pred_path), "target": str(hr_path)}

    def checkpoint_payload(model: nn.Module, ema: EMA | None, optimizer: torch.optim.Optimizer, epoch: int, best_val_psnr: float, row: dict[str, Any]) -> dict[str, Any]:
        payload: dict[str, Any] = {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_psnr": float(best_val_psnr),
            "metrics": row,
            "config": asdict(cfg),
        }
        if ema is not None:
            payload["ema_state_dict"] = ema.shadow
        return payload

    def save_checkpoint(path: Path, payload: dict[str, Any]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload, path)

    def load_checkpoint_for_eval(path: Path, device: torch.device) -> nn.Module:
        loaded = FSRCNNResidualNoUpscale().to(device)
        checkpoint = torch.load(path, map_location=device)
        state = checkpoint.get("ema_state_dict") or checkpoint["model_state_dict"]
        loaded.load_state_dict(state, strict=True)
        loaded.eval()
        return loaded

    model = FSRCNNResidualNoUpscale().to(device)
    eval_model = FSRCNNResidualNoUpscale().to(device)
    param_count = count_parameters(model)
    op_counts = operator_audit(model)
    dummy = torch.zeros(1, 3, cfg.eval_size, cfg.eval_size, device=device)
    with torch.no_grad():
        dummy_out = model(dummy)
    model_summary = {
        "model_id": cfg.model_id,
        "params": param_count,
        "operator_audit": op_counts,
        "input_shape": tuple(dummy.shape),
        "output_shape": tuple(dummy_out.shape),
        "architecture": model.architecture_summary(),
    }
    print(model_summary)
    assert param_count == 133_227, param_count
    assert tuple(dummy_out.shape) == (1, 3, cfg.eval_size, cfg.eval_size)
    assert set(op_counts).issubset({"Conv2d", "LeakyReLU"}), op_counts

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = make_grad_scaler(cfg.use_amp and cuda_enabled)
    ema = EMA(model, cfg.ema_decay) if cfg.use_ema else None
    best_val_psnr = float("-inf")
    start_epoch = 0
    metrics_header = ["epoch", "epoch_time_sec", "lr", "train_loss", "train_psnr", "val_psnr", "input_psnr", "delta_psnr", "best_val_psnr"]
    best_ckpt_path = checkpoints_dir / "best.pth"
    last_ckpt_path = checkpoints_dir / "last.pth"

    resume_checkpoint = torch.load(resume_checkpoint_path, map_location=device)
    model.load_state_dict(resume_checkpoint["model_state_dict"], strict=True)
    if ema is not None and resume_checkpoint.get("ema_state_dict"):
        ema.shadow = {k: v.detach().clone().to(device) for k, v in resume_checkpoint["ema_state_dict"].items()}
        ema.copy_to(eval_model)
    else:
        eval_model.load_state_dict(model.state_dict(), strict=True)
    if cfg.load_optimizer_state and resume_checkpoint.get("optimizer_state_dict"):
        optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    start_epoch = int(resume_checkpoint.get("epoch", 0))
    best_val_psnr = float(resume_checkpoint.get("best_val_psnr", float("-inf")))
    if start_epoch >= cfg.epochs:
        raise ValueError(f"Resume checkpoint epoch {start_epoch} is already >= target epochs {cfg.epochs}")

    resume_info = {
        "enabled": True,
        "checkpoint_path": str(resume_checkpoint_path),
        "checkpoint_epoch": start_epoch,
        "target_total_epochs": cfg.epochs,
        "load_optimizer_state": cfg.load_optimizer_state,
        "optimizer_state_loaded": bool(cfg.load_optimizer_state and resume_checkpoint.get("optimizer_state_dict")),
        "best_val_psnr_at_resume": best_val_psnr,
    }
    print(resume_info)
    append_event({"status": "resume_loaded", "checkpoint_epoch": start_epoch, "target_total_epochs": cfg.epochs})
    write_json(latest_status_path, {"status": "resume_loaded", "run_root": str(run_root), "resume": resume_info})
    commit_run_volume("resume_loaded")

    # Seed this run's best checkpoint from the resume state so ONNX export has
    # a valid best.pth even if the short aggressive continuation does not improve.
    initial_resume_metrics = resume_checkpoint.get("metrics", {"epoch": start_epoch, "best_val_psnr": best_val_psnr})
    initial_best_payload = checkpoint_payload(model, ema, optimizer, start_epoch, best_val_psnr, initial_resume_metrics)
    save_checkpoint(best_ckpt_path, initial_best_payload)

    lr_schedule_info = {
        "type": "warmup_cosine_restart_from_resume_epoch",
        "base_lr": cfg.lr,
        "min_lr": cfg.min_lr,
        "warmup_epochs": cfg.warmup_epochs,
        "schedule_start_epoch": start_epoch,
        "target_total_epochs": cfg.epochs,
    }
    write_json(run_manifest_path, {
        "config": asdict(cfg),
        "data": data_summary,
        "model": model_summary,
        "resume": resume_info,
        "lr_schedule": lr_schedule_info,
        "start_epoch": start_epoch,
        "target_total_epochs": cfg.epochs,
    })

    history: dict[str, Any] = {"epochs": [], "resume": resume_info, "lr_schedule": lr_schedule_info}
    with metrics_csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=metrics_header)
        writer.writeheader()
    commit_run_volume("history_initialized")

    def train_one_epoch(epoch: int, epoch_lr: float) -> dict[str, float]:
        model.train()
        loss_meter = 0.0
        psnr_meter = 0.0
        count = 0
        for batch in train_loader:
            lr = batch["lr"].to(device, non_blocking=cuda_enabled)
            hr = batch["hr"].to(device, non_blocking=cuda_enabled)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context(device.type, cfg.use_amp and cuda_enabled):
                pred = model(lr)
                loss = compute_loss(pred, lr, hr)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if ema is not None:
                ema.update(model)
            loss_meter += float(loss.detach().item())
            psnr_meter += batch_psnr(pred.detach().clamp(0.0, 1.0), hr)
            count += 1
        return {"train_loss": loss_meter / max(count, 1), "train_psnr": psnr_meter / max(count, 1)}

    for epoch in range(start_epoch, cfg.epochs):
        train_ds.set_epoch(epoch)
        epoch_lr = lr_at_epoch(
            epoch,
            base_lr=cfg.lr,
            min_lr=cfg.min_lr,
            warmup_epochs=cfg.warmup_epochs,
            total_epochs=cfg.epochs,
            schedule_start_epoch=start_epoch,
        )
        for group in optimizer.param_groups:
            group["lr"] = epoch_lr
        epoch_start = time.time()
        train_metrics = train_one_epoch(epoch, epoch_lr)
        if ema is not None:
            ema.copy_to(eval_model)
        else:
            eval_model.load_state_dict(model.state_dict(), strict=True)
        val_metrics = evaluate_metrics(eval_model, val_loader, device)
        best_val_psnr = max(best_val_psnr, val_metrics["val_psnr"])
        row = {
            "epoch": epoch + 1,
            "epoch_time_sec": round(time.time() - epoch_start, 2),
            "lr": epoch_lr,
            "train_loss": round(train_metrics["train_loss"], 6),
            "train_psnr": round(train_metrics["train_psnr"], 4),
            "val_psnr": round(val_metrics["val_psnr"], 4),
            "input_psnr": round(val_metrics["input_psnr"], 4),
            "delta_psnr": round(val_metrics["delta_psnr"], 4),
            "best_val_psnr": round(best_val_psnr, 4),
        }
        history["epochs"].append(row)
        payload = checkpoint_payload(model, ema, optimizer, epoch + 1, best_val_psnr, row)
        save_checkpoint(last_ckpt_path, payload)
        if val_metrics["val_psnr"] >= best_val_psnr:
            save_checkpoint(best_ckpt_path, payload)
        with metrics_csv_path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=metrics_header)
            writer.writerow(row)
        write_json(history_json_path, history)
        write_json(latest_status_path, {"status": "training", "latest": row, "run_root": str(run_root), "metrics_csv": str(metrics_csv_path), "history_json": str(history_json_path)})
        append_event({"status": "epoch_complete", **row})
        commit_run_volume(f"epoch_{epoch + 1}")
        print(row)


    export_model = load_checkpoint_for_eval(best_ckpt_path, device)
    validation_summary = evaluate_metrics(export_model, val_loader, device)
    preview_info = write_val_preview(export_model, val_loader, preview_dir, device)
    print({"validation_summary": validation_summary, "best_checkpoint_path": str(best_ckpt_path), "last_checkpoint_path": str(last_ckpt_path), "preview": preview_info})

    def load_or_create_calibration_summary(calibration_dir: Path, pairs: list[tuple[Path, Path, str]], count: int, eval_size: int) -> dict[str, Any]:
        manifest_path = calibration_dir / "manifest.json"
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            item_count = int(manifest.get("count") or len(manifest.get("items", [])))
            if item_count <= 0 or manifest.get("derived_from_training") is not True:
                raise ValueError(f"Invalid training-derived calibration manifest: {manifest_path}")
            return {
                "calibration_dir": str(calibration_dir),
                "manifest_path": str(manifest_path),
                "count": item_count,
                "source": manifest.get("source"),
                "derived_from_training": True,
                "reused_existing_dataset": True,
            }

        calibration_dir.mkdir(parents=True, exist_ok=True)
        items: list[dict[str, Any]] = []
        for index, (lr_path, _, name) in enumerate(pairs[:count]):
            image = Image.open(lr_path).convert("RGB")
            if image.size != (eval_size, eval_size):
                image = image.resize((eval_size, eval_size), BICUBIC_RESAMPLE)
            image_path = calibration_dir / f"{index:03d}_{Path(name).name}.png"
            image.save(image_path)
            items.append({
                "index": index,
                "name": name,
                "source_lr": str(lr_path),
                "image_path": str(image_path),
                "derived_from_training": True,
            })
        manifest = {
            "count": len(items),
            "source": "training_pairs",
            "eval_size": eval_size,
            "derived_from_training": True,
            "items": items,
        }
        write_json(manifest_path, manifest)
        return {
            "calibration_dir": str(calibration_dir),
            "manifest_path": str(manifest_path),
            "count": len(items),
            "source": "training_pairs",
            "derived_from_training": True,
            "reused_existing_dataset": False,
        }

    conversion_script = """#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import shlex
import subprocess
from pathlib import Path


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description='Lab 3 ONNX-to-MXQ handoff script for FSRCNN residual.')
    parser.add_argument('--onnx', type=Path, required=True)
    parser.add_argument('--calibration-dir', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--command-template', default='', help='Optional command with {onnx}, {calibration}, {output} placeholders.')
    parser.add_argument('--dry-run', action='store_true')
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    args.onnx = args.onnx.resolve()
    args.calibration_dir = args.calibration_dir.resolve()
    args.output = args.output.resolve()
    if not args.onnx.exists():
        raise FileNotFoundError(f'ONNX file not found: {args.onnx}')
    if not args.calibration_dir.exists():
        raise FileNotFoundError(f'Calibration directory not found: {args.calibration_dir}')
    payload = {
        'onnx': str(args.onnx),
        'calibration_dir': str(args.calibration_dir),
        'output': str(args.output),
        'status': 'handoff_only',
        'command_template': args.command_template,
    }
    if args.command_template:
        command = args.command_template.format(
            onnx=shlex.quote(str(args.onnx)),
            calibration=shlex.quote(str(args.calibration_dir)),
            output=shlex.quote(str(args.output)),
        )
        payload['command'] = command
        if not args.dry_run:
            completed = subprocess.run(command, shell=True, check=False, capture_output=True, text=True)
            payload.update({
                'status': 'completed' if completed.returncode == 0 else 'failed',
                'returncode': completed.returncode,
                'stdout': completed.stdout,
                'stderr': completed.stderr,
            })
    args.output.parent.mkdir(parents=True, exist_ok=True)
    handoff_path = args.output.with_suffix('.handoff.json')
    handoff_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print(json.dumps(payload, indent=2, sort_keys=True))


if __name__ == '__main__':
    main()
"""

    calibration_summary = load_or_create_calibration_summary(calibration_dir, train_pairs, cfg.calibration_count, cfg.eval_size)
    onnx_path = exports_dir / f"{cfg.model_id}.onnx"
    mxq_target_path = exports_dir / f"{cfg.model_id}.mxq"
    conversion_script_path = exports_dir / "convert_fsrcnn_residual_mxq.py"
    conversion_script_path.write_text(conversion_script, encoding="utf-8")

    cpu_export_model = load_checkpoint_for_eval(best_ckpt_path, torch.device("cpu")).cpu().eval()
    dummy = torch.randn(1, 3, cfg.eval_size, cfg.eval_size)
    torch.onnx.export(
        cpu_export_model,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        dynamo=False,
    )
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    node_ops = [node.op_type for node in onnx_model.graph.node]
    op_histogram = {op: node_ops.count(op) for op in sorted(set(node_ops))}
    onnx_summary = {
        "onnx_path": str(onnx_path),
        "checked": True,
        "graph_op_count": len(node_ops),
        "op_histogram": op_histogram,
        "input_shape": [1, 3, cfg.eval_size, cfg.eval_size],
        "output_shape": [1, 3, cfg.eval_size, cfg.eval_size],
    }
    print(onnx_summary)

    parity_result = {"attempted": False, "onnxruntime_available": False, "max_abs_diff": None, "mean_abs_diff": None}
    if ort is not None:
        parity_result["attempted"] = True
        parity_result["onnxruntime_available"] = True
        with torch.no_grad():
            torch_out = cpu_export_model(dummy).detach().numpy()
        ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = ort_session.run(None, {"input": dummy.numpy()})[0]
        diff = np.abs(torch_out - ort_out)
        parity_result.update({"max_abs_diff": float(diff.max()), "mean_abs_diff": float(diff.mean())})
    else:
        print("onnxruntime not installed; skipping CPU parity check")
    print(parity_result)

    mxq_command = f"python {conversion_script_path} --onnx {onnx_path} --calibration-dir {calibration_dir} --output {mxq_target_path} --dry-run"
    mxq_handoff = {
        "conversion_script": str(conversion_script_path),
        "onnx_input_path": str(onnx_path),
        "calibration_dir": str(calibration_dir),
        "mxq_target_path": str(mxq_target_path),
        "command": mxq_command,
        "status": "handoff_only",
    }

    artifacts = {
        "best_checkpoint": str(best_ckpt_path),
        "last_checkpoint": str(last_ckpt_path),
        "onnx": str(onnx_path),
        "calibration_manifest": str(calibration_dir / "manifest.json"),
        "conversion_script": str(conversion_script_path),
        "mxq_target": str(mxq_target_path),
        "metrics_csv": str(metrics_csv_path),
        "history_json": str(history_json_path),
        "summary_json": str(summary_path),
        "run_manifest": str(run_manifest_path),
        "latest_status_json": str(latest_status_path),
        "events_jsonl": str(events_jsonl_path),
    }
    gates = {
        "shape_contract_pass": model_summary["input_shape"] == (1, 3, 256, 256) and model_summary["output_shape"] == (1, 3, 256, 256),
        "operator_audit_pass": set(op_counts).issubset({"Conv2d", "LeakyReLU"}),
        "onnx_pass": onnx_summary["checked"] and (not parity_result["attempted"] or parity_result["max_abs_diff"] is not None and parity_result["max_abs_diff"] < 1e-3),
        "calibration_pass": calibration_summary["derived_from_training"] is True and calibration_summary["count"] > 0,
        "mxq_handoff_pass": conversion_script_path.exists() and mxq_handoff["status"] == "handoff_only",
    }
    gates["submission_chain_pass"] = all(gates.values())

    summary = {
        "backend": "modal",
        "run_root": str(run_root),
        "config": asdict(cfg),
        "data": data_summary,
        "model": model_summary,
        "resume": resume_info,
        "lr_schedule": lr_schedule_info,
        "training": {
            "start_epoch": start_epoch,
            "target_total_epochs": cfg.epochs,
            "epochs_completed_this_run": max(0, cfg.epochs - start_epoch),
            "latest_metrics": history["epochs"][-1] if history["epochs"] else initial_resume_metrics,
        },
        "validation": validation_summary,
        "preview": preview_info,
        "onnx": onnx_summary,
        "onnx_parity": parity_result,
        "calibration": calibration_summary,
        "mxq_handoff": mxq_handoff,
        "artifacts": artifacts,
        "gates": gates,
        "execution": {"backend": "modal", "final_status": "completed"},
    }
    write_json(summary_path, summary)
    write_json(latest_status_path, {"status": "completed", "summary_path": str(summary_path), "artifacts": artifacts, "gates": gates})
    append_event({"status": "completed", "summary_path": str(summary_path), "submission_chain_pass": gates["submission_chain_pass"]})
    commit_run_volume("completed")
    print({
        "summary_path": str(summary_path),
        "best_checkpoint": str(best_ckpt_path),
        "last_checkpoint": str(last_ckpt_path),
        "onnx": str(onnx_path),
        "calibration_manifest": str(calibration_dir / "manifest.json"),
        "conversion_script": str(conversion_script_path),
        "mxq_target": str(mxq_target_path),
    })
    return summary


## Export


In [ ]:
print({
    "export_contract": "ONNX input/output fixed at (1, 3, 256, 256)",
    "calibration_source": "training LR pairs only",
    "mxq_handoff": "convert_fsrcnn_residual_mxq.py generated in the run exports directory",
})


##  Modal App Launcher


In [ ]:

def _modal_image() -> modal.Image:
    return modal.Image.debian_slim(python_version="3.13").pip_install(
        "torch==2.5.1",
        "numpy==2.1.3",
        "pillow==11.0.0",
        "onnx==1.20.1",
        "onnxruntime==1.20.1",
    )


def _run_modal_cli(args: list[str]) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, check=False, capture_output=True, text=True, cwd=str(PROJECT_ROOT))


def _volume_path_exists(volume_name: str, remote_path: str) -> bool:
    completed = _run_modal_cli(["modal", "volume", "ls", volume_name, remote_path])
    return completed.returncode == 0


REQUIRED_L3_VOLUME_PATHS = [
    "/Data/LR_HR",
    "/Data/L3_LR",
    "/Data/L3_HR_valid",
    "/Data/L3_LR_valid",
]


def _remote_runs_path(remote_path: str) -> str:
    if remote_path.startswith("/mnt/lab3-runs"):
        remote_path = remote_path[len("/mnt/lab3-runs") :]
    return remote_path if remote_path.startswith("/") else f"/{remote_path}"


def _assert_remote_path_exists(volume_name: str, remote_path: str, label: str) -> None:
    completed = _run_modal_cli(["modal", "volume", "ls", volume_name, remote_path])
    if completed.returncode != 0:
        detail = completed.stderr.strip() or completed.stdout.strip() or "not found"
        raise FileNotFoundError(f"Missing {label} in Modal volume {volume_name}: {remote_path}. {detail}")


def sync_data_volume_if_needed() -> dict[str, Any]:
    if not LOCAL_DATA_ROOT.exists():
        raise FileNotFoundError(f"Local Data directory not found for Modal sync: {LOCAL_DATA_ROOT}")
    missing_l3_paths = [path for path in REQUIRED_L3_VOLUME_PATHS if not _volume_path_exists(MODAL_DATA_VOLUME, path)]
    if not FORCE_DATA_SYNC and not missing_l3_paths:
        return {"status": "reused_existing_l3", "volume_name": MODAL_DATA_VOLUME, "verified_paths": REQUIRED_L3_VOLUME_PATHS}
    if not SYNC_DATA_TO_VOLUME and missing_l3_paths:
        raise FileNotFoundError({"missing_l3_paths": missing_l3_paths, "hint": "Set LAB3_FSRCNN_SYNC_DATA=true or upload the L3 directories to lab3-data."})
    uploads = {
        "LR_HR": "/Data/LR_HR",
        "L3_LR": "/Data/L3_LR",
        "L3_HR_valid": "/Data/L3_HR_valid",
        "L3_LR_valid": "/Data/L3_LR_valid",
    }
    synced: list[str] = []
    for local_name, remote_path in uploads.items():
        if not FORCE_DATA_SYNC and remote_path not in missing_l3_paths:
            continue
        local_path = LOCAL_DATA_ROOT / local_name
        if not local_path.exists():
            raise FileNotFoundError(f"Local L3 data directory not found: {local_path}")
        completed = _run_modal_cli(["modal", "volume", "put", MODAL_DATA_VOLUME, str(local_path), remote_path, "--force"])
        if completed.returncode != 0:
            raise RuntimeError(completed.stderr.strip() or completed.stdout.strip() or f"modal volume put failed for {local_path}")
        synced.append(remote_path)
    for required_path in REQUIRED_L3_VOLUME_PATHS:
        _assert_remote_path_exists(MODAL_DATA_VOLUME, required_path, "L3 data path")
    return {"status": "synced_l3" if synced else "reused_existing_l3", "volume_name": MODAL_DATA_VOLUME, "synced_paths": synced, "verified_paths": REQUIRED_L3_VOLUME_PATHS}


def sync_submission_resume_checkpoint_if_needed(config: dict[str, Any]) -> dict[str, Any]:
    if not LOCAL_SUBMISSION_LAST_PTH.is_file():
        raise FileNotFoundError(
            f"Local submission checkpoint not found: {LOCAL_SUBMISSION_LAST_PTH}. "
            "Place epoch-8080 weights at experiments/FSRCNNResidual/submission/last.pth."
        )
    resume_path = config["resume_checkpoint_path"]
    if not resume_path.startswith("/mnt/lab3-runs"):
        raise ValueError(f"resume_checkpoint_path must be under /mnt/lab3-runs, got {resume_path}")
    checkpoint_key = _remote_runs_path(resume_path)
    if not FORCE_RESUME_UPLOAD and _volume_path_exists(MODAL_RUNS_VOLUME, checkpoint_key):
        return {
            "status": "reused_existing_resume",
            "volume_path": checkpoint_key,
            "local_checkpoint": str(LOCAL_SUBMISSION_LAST_PTH),
        }
    completed = _run_modal_cli(
        [
            "modal",
            "volume",
            "put",
            MODAL_RUNS_VOLUME,
            str(LOCAL_SUBMISSION_LAST_PTH),
            checkpoint_key,
            "--force",
        ]
    )
    if completed.returncode != 0:
        raise RuntimeError(
            completed.stderr.strip() or completed.stdout.strip() or "modal volume put resume checkpoint failed"
        )
    _assert_remote_path_exists(MODAL_RUNS_VOLUME, checkpoint_key, "FSRCNN resume checkpoint")
    return {
        "status": "uploaded_resume",
        "volume_path": checkpoint_key,
        "local_checkpoint": str(LOCAL_SUBMISSION_LAST_PTH),
    }


def preflight_modal_prerequisites(config: dict[str, Any]) -> dict[str, Any]:
    for required_path in REQUIRED_L3_VOLUME_PATHS:
        _assert_remote_path_exists(MODAL_DATA_VOLUME, required_path, "L3 data path")
    checkpoint_path = _remote_runs_path(config["resume_checkpoint_path"])
    _assert_remote_path_exists(MODAL_RUNS_VOLUME, checkpoint_path, "FSRCNN resume checkpoint")
    return {"l3_data_verified": REQUIRED_L3_VOLUME_PATHS, "resume_checkpoint_verified": checkpoint_path}


def sync_run_from_volume(remote_run_root: str) -> Path:
    remote_path = remote_run_root.replace("/mnt/lab3-runs", "")
    if not remote_path.startswith("/"):
        remote_path = "/" + remote_path
    LOCAL_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    completed = _run_modal_cli(["modal", "volume", "get", MODAL_RUNS_VOLUME, remote_path, str(LOCAL_RUNS_ROOT), "--force"])
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip() or "modal volume get failed")
    return LOCAL_RUNS_ROOT / Path(remote_path).name


data_volume = modal.Volume.from_name(MODAL_DATA_VOLUME, create_if_missing=True)
runs_volume = modal.Volume.from_name(MODAL_RUNS_VOLUME, create_if_missing=True)
app = modal.App(APP_NAME)


@app.function(
    image=_modal_image(),
    gpu=MODAL_GPU,
    serialized=True,
    timeout=MODAL_TIMEOUT_MINUTES * 60,
    volumes={"/mnt/lab3-data": data_volume, "/mnt/lab3-runs": runs_volume},
    name=FUNCTION_NAME,
)
def run_remote(config: dict[str, Any]) -> dict[str, Any]:
    summary = run_fsrcnn_pipeline(config)
    runs_volume.commit()
    return summary


def execute_modal_run(config: dict[str, Any]) -> dict[str, Any]:
    data_sync = sync_data_volume_if_needed()
    resume_sync = sync_submission_resume_checkpoint_if_needed(config)
    preflight = preflight_modal_prerequisites(config)
    print(json.dumps({"preflight": preflight, "data_sync": data_sync, "resume_sync": resume_sync, "modal_detach": MODAL_DETACH}, indent=2, sort_keys=True))
    with app.run(detach=MODAL_DETACH):
        call = run_remote.spawn(config)
        while True:
            try:
                summary = call.get(timeout=POLL_INTERVAL_SECONDS)
                break
            except TimeoutError:
                print({"status": "still_running", "timestamp": time.strftime("%Y-%m-%d %H:%M:%S %Z")})
    local_run_root = sync_run_from_volume(summary["run_root"])
    summary.setdefault("execution", {})["data_volume_sync"] = data_sync
    summary["execution"]["resume_checkpoint_sync"] = resume_sync
    summary["execution"]["preflight"] = preflight
    summary["execution"]["synced_local_run_root"] = str(local_run_root)
    print(json.dumps(summary, indent=2, sort_keys=True))
    return summary


## Final Summary


In [ ]:

if RUN_MODAL_SUBMISSION:
    summary = execute_modal_run(RUN_CONFIG)
    final_paths = {
        "run_root": summary["run_root"],
        "synced_local_run_root": summary.get("execution", {}).get("synced_local_run_root"),
        "best_pth": summary["artifacts"]["best_checkpoint"],
        "last_pth": summary["artifacts"]["last_checkpoint"],
        "onnx": summary["artifacts"]["onnx"],
        "calibration_manifest": summary["artifacts"]["calibration_manifest"],
        "conversion_script": summary["artifacts"]["conversion_script"],
        "mxq_target": summary["artifacts"]["mxq_target"],
        "summary_json": summary["artifacts"]["summary_json"],
        "run_manifest": summary["artifacts"]["run_manifest"],
        "latest_status_json": summary["artifacts"].get("latest_status_json"),
        "events_jsonl": summary["artifacts"].get("events_jsonl"),
    }
    print(json.dumps(final_paths, indent=2, sort_keys=True))
else:
    print("RUN_MODAL_SUBMISSION is false; set LAB3_FSRCNN_RUN_MODAL=true to launch the Modal app.")
